# Waterberry Farms: a realistic benchmark for multi-robot informative path planning

This notebook contains the code for generating the pictures and graphs for the paper "Waterberry Farms: A Novel Benchmark For Informative Path Planning" by
Samuel Matloob, Partha P. Datta, O. Patrick Kreidl, Ayan Dutta, Swapnoneel Roy, Ladislau Bölöni. (https://arxiv.org/abs/2305.06243)



It illustrates various ways of running a __one day__ experiment on a __static environment__ with a __single robot__. The experiments compare estimator/path planner pairs. The path planners are simple static examples (lawnmower, spiral, random waypoint) and they are not intended to be state of the art. 

In [ ]:
# allow imports from the main source directory
import sys
sys.path.append('../..')
import logging
logging.basicConfig(level=logging.WARNING)

import pathlib
import pickle
import gzip as compress
import copy
import matplotlib.pyplot as plt
# configuring the fonts such that no Type 3 fonts are used
# requirement for ICRA
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams["figure.autolayout"] = True

import numpy as np
import timeit
import warnings

from exp_run_config import Config
from water_berry_farm import WBF_IM_DiskEstimator, WBF_IM_GaussianProcess, WBF_Score_WeightedAsymmetric
from wbf_helper import create_wbf, create_wbfe, get_geometry, precompute_environment
from wbf_simulate import save_simulation_results, simulate_1day
from wbf_figures import end_of_day_graphs, hook_create_pictures
from policy import FollowPathPolicy, RandomWaypointPolicy
from path_generators import find_fixed_budget_lawnmower, find_fixed_budget_spiral, generate_lawnmower, get_path_length
from robot import Robot
Config.PROJECTNAME = "WaterBerryFarms"

logging.getLogger().setLevel(logging.WARNING)

# all the experiments results etc. should go into this directory
benchmark_dir = pathlib.Path(Config().get_results_path(), "Benchmark-Paper")
benchmark_dir.mkdir(parents=True, exist_ok=True)

## General settings
The choices used here apply to all the components and might make them faster

In [ ]:
# the environment exp/run used for each farm type
environment_runs = {"Miniberry-10": "benchmark-miniberry-10", "Miniberry-30": "benchmark-miniberry-30",
                    "Miniberry-100": "benchmark-miniberry-100", "Waterberry": "benchmark-waterberry"}
# the farm type of the one day experiments
experiment_typename = "Miniberry-30"
timesteps = 500 # 1000
time_start_environment = 6
# change this to a lower number to run the estimator more often
estimator_interval = 10
# if force_run is false, we do not re-run an already run experiment
force_run = False

def run_1day(policy, estimator, evaluator):
    """Runs a one day experiment with a single robot on the experiment farm and saves the results into the 
    benchmark directory. If the results already exist and force_run is false, it loads them instead."""
    resultsfile = pathlib.Path(benchmark_dir, f"res_{policy.name}-{estimator.name}.pickle")
    print(resultsfile)
    if resultsfile.exists() and not force_run:
        with compress.open(resultsfile, "rb") as f:
            return pickle.load(f)
    _, environment = create_wbfe(Config().get_experiment("environment", environment_runs[experiment_typename]))
    environment.proceed(time_start_environment)
    robot = Robot("Rob", 0, 0, 0)
    robot.assign_policy(policy)
    # add after_timestep = hook_create_pictures to generate pictures. WARNING: takes longer!
    results = simulate_1day(environment=environment, robots=[robot], estimator=estimator, evaluator=evaluator,
                            timesteps=timesteps, estimator_interval=estimator_interval)
    results.update({"wbfe": environment, "policy-name": policy.name, "estimator-name": estimator.name,
                    "results-basedir": benchmark_dir})
    save_simulation_results(resultsfile, results)
    return results

# Precompute environments
* Precompute the evaluation of the environments. This allows the following simulations to run faster and to use the same environment for different path planning algorithms. 

* Each farm type has its own environment exp/run (`data/expruns/environment/benchmark-*.yaml`), which precalculates 50 days of evolution. This is sufficient for the standard benchmark settings. An environment is precomputed only once: it is skipped if an earlier precomputation completed. __Delete its directory `environment/benchmark-*` in the results path if you want to recompute it.__

* Waterberry is commented out of the list below, as precomputing it takes about __30 minutes__ on a fast machine. The cells below that use Waterberry precompute it when needed.

In [ ]:
typenames = ["Miniberry-10", "Miniberry-30", "Miniberry-100"]
# typenames = ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]
for typename in typenames:
    precompute_environment(environment_runs[typename])

# Generate pictures of the geometry. 
Generates pictures for the geometry of the various benchmark configurations (Miniberry-10,30,100 and Waterberry), and saves them into files $data/geometry_Miniberry-10.pdf etc. Only the farm geometry is created, so this does not need the precomputed environments.

In [ ]:
typenames_1 = ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]
for typename in typenames_1:
    wbf = create_wbf({"typename": typename})
    fig, ax = plt.subplots(1, figsize=(5,4))
    wbf.visualize(ax)
    fig.tight_layout()
    plt.savefig(pathlib.Path(benchmark_dir, f"geometry_{typename}.pdf"))
    plt.savefig(pathlib.Path(benchmark_dir, f"geometry_{typename}.jpg"))

## Compare movement policies with the same estimator
Run one day experiments for comparing different policies with the same estimator. Visualize the results for each policy.

In [ ]:
def create_policies(geom):
    policies = []
    # a random waypoint policy
    geo = copy.copy(geom)
    policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)  
    policy.name = "RandomWaypoint-Seed0"
    policies.append(policy)

    # a lawnmower policy that covers the target area in one day uniformly
    geo = copy.copy(geom)
    path = find_fixed_budget_lawnmower([0,0], geo["xmin"], geo["xmax"], geo["ymin"], geo["ymax"], geo["velocity"], time = geo["timesteps-per-day"])
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True)
    policy.name = "FixedBudgetLawnmower"
    policies.append(policy)

    # a lawnmower policy that covers the tomatoes more densely than the strawberries
    geo = copy.copy(geom)
    height = geo["ymax"] - geo["ymin"]
    time = float(geo["timesteps-per-day"])
    time_safety = time / 10.0
    # the strawberry leg gets a third of the time, but at least enough for a one-wind lawnmower 
    # from any corner of the tomato area where the tomato leg may end
    strawberry_area = [geo["xmin"], geo["xmax"], geo["ymin"], geo["ymin"] + height // 2]
    corners = [[x, y] for x in [geo["xmin"], geo["xmax"]] for y in [geo["ymin"] + height // 2, geo["ymin"] + height]]
    time_strawberry_min = max(get_path_length(generate_lawnmower(*strawberry_area, 1), corner) for corner in corners) / geo["velocity"]
    time_strawberry = max(time / 3.0 - time_safety, time_strawberry_min)
    time_tomato = time - time_strawberry - time_safety
    path1 = find_fixed_budget_lawnmower([0, 0], geo["xmin"], geo["xmax"], geo["ymin"] + height//2, geo["ymin"] + height, geo["velocity"], time_tomato)
    path2 = find_fixed_budget_lawnmower(path1[-1], geo["xmin"], geo["xmax"], geo["ymin"], geo["ymin"] + height // 2, geo["velocity"], time_strawberry)
    # second path is supposed to start from the last of first
    path = np.append(path1, path2, axis=0)           
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True)            
    policy.name = "AdaptiveLawnMower"
    policies.append(policy)

    # a spiral policy 
    geo = copy.copy(geom)
    path = find_fixed_budget_spiral([0,0], x_min = geo["xmin"], x_max = geo["xmax"], y_min = geo["ymin"], y_max = geo["ymax"], velocity = geo["velocity"], time = geo["timesteps-per-day"])
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True) 
    policy.name = "Spiral"    
    policies.append(policy)
    return policies

In [ ]:
geo = get_geometry(experiment_typename)
for policy in create_policies(geo):
    estimator = WBF_IM_DiskEstimator(geo["width"], geo["height"])
    results = run_1day(policy, estimator, WBF_Score_WeightedAsymmetric())
    end_of_day_graphs(results, f"daily-summary-{results['policy-name']}-{results['estimator-name']}.pdf", plot_uncertainty = False)

## Compare estimators in experiments
Run experiments to compare the estimators - the adaptive disk (AD) vs the Gaussian Process (GP). The Gaussian process is much slower, so it will take a quite longer time.

In [ ]:
# cleaning up the warnings from the GP estimator
warnings.simplefilter("ignore")

geo = get_geometry(experiment_typename)
for estimator_name in ["AD", "GP", "DE-10"]:
# for estimator_name in ["GP"]:
    policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)  
    policy.name = f"RandomWaypoint-Seed0-{estimator_name}"
    if estimator_name == "AD":
        estimator = WBF_IM_DiskEstimator(geo["width"], geo["height"])
    elif estimator_name == "GP":
        estimator = WBF_IM_GaussianProcess(geo["width"], geo["height"])        
    elif estimator_name == "DE-10":
        estimator = WBF_IM_DiskEstimator(geo["width"], geo["height"], disk_radius=10)        
    estimator.name = estimator_name
    results = run_1day(policy, estimator, WBF_Score_WeightedAsymmetric())
    end_of_day_graphs(results, f"daily-summary-{results['policy-name']}.pdf", plot_uncertainty = False)

## Compare the performance of various combinations of path planner and estimator

In [ ]:
# gather all the results. As I am reading them in, generate the graphs.
allresults = []
for a in sorted(benchmark_dir.iterdir()):
    if a.name.startswith("res_"):
        print(a)
        with compress.open(a, "rb") as f:
            results = pickle.load(f)
            allresults.append(results)

fig, ax_scores = plt.subplots(1, figsize=(6,4))
for results in allresults:
    events = results["score-events"]
    ax_scores.plot([event["timestep"] for event in events], [event["score"] for event in events],
                   label = f'{results["policy-name"]}+{results["estimator-name"]}')
ax_scores.set_ylim(top=0)
ax_scores.set_xlabel("Time")
ax_scores.set_ylabel("Score")
ax_scores.set_title("Scores")
ax_scores.legend()
fig.tight_layout()

## Compare the results of different score functions
Verifying that different scoring functions are called

In [ ]:
geo = get_geometry(experiment_typename)

# create a set of score functions
scores = {}
scores["default"] = WBF_Score_WeightedAsymmetric()
scores["only-strawberry"] = WBF_Score_WeightedAsymmetric(strawberry_importance=1.0, strawberry_negative_importance=10.0, tomato_importance=0.0, tomato_negative_importance=10.0, soil_importance=0.0)
scores["only-tomato"] = WBF_Score_WeightedAsymmetric(strawberry_importance=0.0, strawberry_negative_importance=10.0, tomato_importance=1.0, tomato_negative_importance=10.0, soil_importance=0.0)
scores["only-soil"] = WBF_Score_WeightedAsymmetric(strawberry_importance=0.0, strawberry_negative_importance=10.0, tomato_importance=0.0, tomato_negative_importance=10.0, soil_importance=1.0)

for scorename, evaluator in scores.items():
    # the same random waypoint path for every score function
    policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)  
    policy.name = f"random-{scorename}"
    estimator = WBF_IM_DiskEstimator(geo["width"], geo["height"])
    results = run_1day(policy, estimator, evaluator)
    end_of_day_graphs(results, f"daily-summary-{results['policy-name']}-{results['estimator-name']}.pdf", plot_uncertainty = False)

# Comparing the computational cost of different estimators
This code calculates the computational cost for different estimators (current GP and adaptive disk). This code takes about 10 min to run. 

In [ ]:
from logging import getLogger

logging.getLogger().setLevel(logging.INFO)
warnings.simplefilter("ignore")

typenames = ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]

file = pathlib.Path(benchmark_dir, "timecomp")
totaltime = 500 # the time we extend if 
timestep = 10 # at what timesteps we do a measurement
stop_at = 10.0 # if a single estimate takes longer than this, we stop

if file.exists():
    with open(file, "rb") as f:
        res = pickle.load(f)
else:
    res = {}
    random = np.random.default_rng(0)
    for estimator in ["AD", "GP"]:
        for typename in typenames:
            # break out, as GP Waterberry 10 is 160 seconds
            if estimator == "GP" and typename == "Waterberry":
                break
            wbf, wbfe = create_wbfe(precompute_environment(environment_runs[typename]))
            wbfe.proceed(time_start_environment)
            if estimator == "AD":
                wbfim = WBF_IM_DiskEstimator(wbf.width, wbf.height)
            else: 
                wbfim = WBF_IM_GaussianProcess(wbf.width, wbf.height)
            ## create some observations and add them...
            times = []
            ts = []
            t = 0
            while t < totaltime:
                for obs in range(timestep):
                    t = t + 1
                    x = int(random.integers(wbf.width))
                    y = int(random.integers(wbf.height))
                    position = [x, y, t] # fixme random position
                    obs = wbfe.get_observation(position)
                    # observations passed to a Waterberry estimator carry the robot that made them
                    obs["robot"] = "random"
                    wbfim.add_observation(obs)
                ## measure the time it takes here
                time = timeit.timeit("wbfim.proceed(1)", number=1,  globals=globals())
                getLogger().info(f"{t} = {time} {estimator} {typename}")
                if time > stop_at:
                    break
                times.append(time)
                ts.append(t)
            res[f"{typename}_{estimator}"] = [ts, times]
    with open(file, "wb") as f:
        pickle.dump(res, f)

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(6,4))

for key in res:
    ax.plot(res[key][0], res[key][1], label = key)
#ax.set_ylim(top=0)
ax.set_xlim(right=700)
ax.set_xlabel("Observations")
ax.set_ylabel("Time (seconds)")
#ax_scores.set_title("Scores")
ax.legend()
fig.tight_layout()
plt.savefig(pathlib.Path(benchmark_dir, "estimator_computational_cost.pdf"))


# Evolution of the environments at a list of specific timepoints

Generates figures for the state of the environment components at various timepoints. The default timepoints are the 1, 5, 10, 15 that are good for visualizing what goes on in the multiday experiments. 

The results are saved into $data/env-dynamics-{typename}-{value} files.

In [ ]:
typenames = ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]
#typenames = ["Miniberry-100"]
values = ["tylcv", "ccr", "soil"]
timepoints = [1, 5, 10, 15]

for value in values:
    for typename in typenames:
        _, wbfe = create_wbfe(precompute_environment(environment_runs[typename]))
        fig, axs = plt.subplots(1, len(timepoints), figsize = (2.5 * len(timepoints), 2.5))
        #fig.suptitle(f"{typename} / {value}")
        ntt = 0
        for t in range(timepoints[-1]+1):
            if t >= timepoints[ntt]:
                if value == "tylcv":
                    val = wbfe.tylcv.value
                elif value == "ccr":
                    val = wbfe.ccr.value
                elif value == "soil":
                    val = wbfe.soil.value
                axs[ntt].imshow(val.T, vmin=0, vmax=1, cmap="gray", origin="lower")
                axs[ntt].set_title(f"t={timepoints[ntt]}")
                ntt = ntt + 1
            wbfe.proceed()
        fig.tight_layout()
        plt.savefig(pathlib.Path(benchmark_dir, f"env-dynamics-{typename}-{value}.pdf"))
        plt.savefig(pathlib.Path(benchmark_dir, f"env-dynamics-{typename}-{value}.jpg"))